In [ ]:
import os
import sys

sys.path.append("/home/justin/code/point-to-pose/")
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
from typing import List, Tuple, Optional, Dict
import glob

# For interactive 3D visualization
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("Note: plotly not available. Will use matplotlib for 3D visualization (less interactive).")

# Open3D for comparison
try:
    import open3d as o3d
    HAS_OPEN3D = True
except ImportError:
    HAS_OPEN3D = False
    print("Note: open3d not available. Open3D normal comparison will be skipped.")

# Import the normal computation function
from point2pose.utils.camera import compute_normals_from_depth

print("Imports successful!")


In [ ]:
# Configure data path
# data_path = "/home/justin/code/Manipulator-Software/data/object_mesh/edamame_box"
# data_path = "/home/justin/code/Manipulator-Software/data/object_mesh/green_bowl"
data_path = "/home/justin/data/HO3D_V3/evaluation/AP10"

# Camera intrinsics (default values - adjust based on your dataset)
# For HO3D, typical values are around:
# fx = fy ≈ 600-700, cx = W/2, cy = H/2
# You can also load from a config file or estimate from the images
DEFAULT_FX = 600.0  # Adjust based on your camera
DEFAULT_FY = 600.0  # Adjust based on your camera

# HO3D depth scale: depth images are uint16, need to multiply by 0.00012498664727900177 to get meters
# Since compute_normals_from_depth divides by depth_factor, we use: 1 / 0.00012498664727900177
HO3D_DEPTH_SCALE = 0.00012498664727900177  # meters per unit
DEFAULT_DEPTH_FACTOR = 1.0 / HO3D_DEPTH_SCALE  # ≈ 8000.0 for HO3D dataset
# For other datasets:
# - RealSense: 1000.0 (depth in mm)
# - Already in meters: 1.0

# Normal computation parameters
MIN_DEPTH = 0.05  # meters
MAX_DEPTH = 1.0   # meters
FILL_MISSING_DEPTH = False  # Set to True to fill missing depth values
WINDOW_SIZE = 3  # For depth filling (odd integer: 3, 5, 7, ...)
MIN_NEIGHBORS = 1  # Minimum valid neighbors for depth filling


In [ ]:
# Function to load RGB images
def load_rgb_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load RGB images from the /rgb subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /rgb subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.jpg', '.jpeg', '.png'])
    
    Returns:
        List[np.ndarray]: List of RGB images as numpy arrays
    """
    if file_extensions is None:
        file_extensions = ['.jpg', '.jpeg', '.png']
    
    rgb_folder = os.path.join(folder_path, 'rgb')
    if not os.path.exists(rgb_folder):
        raise FileNotFoundError(f"RGB folder not found: {rgb_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(rgb_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            img = cv2.imread(file_path)
            if img is not None:
                # Convert BGR to RGB
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                images.append(img_rgb)
    
    return images

# Function to load depth images
def load_depth_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load depth images from the /depth subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /depth subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.png', '.tiff', '.tif'])
    
    Returns:
        List[np.ndarray]: List of depth images as numpy arrays
    """
    if file_extensions is None:
        file_extensions = ['.png', '.tiff', '.tif']
    
    depth_folder = os.path.join(folder_path, 'depth')
    if not os.path.exists(depth_folder):
        raise FileNotFoundError(f"Depth folder not found: {depth_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(depth_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            # Load depth image (usually 16-bit)
            img = cv2.imread(file_path, cv2.IMREAD_ANYDEPTH)
            if img is not None:
                images.append(img)
    
    return images

# Function to load mask images
def load_mask_images(folder_path: str, file_extensions: List[str] = None) -> List[np.ndarray]:
    """
    Load mask images from the /masks subdirectory.
    
    Args:
        folder_path (str): Path to the main folder containing /masks subdirectory
        file_extensions (List[str]): List of file extensions to look for (default: ['.png', '.jpg', '.jpeg'])
    
    Returns:
        List[np.ndarray]: List of mask images as numpy arrays (binary or grayscale)
    """
    if file_extensions is None:
        file_extensions = ['.png', '.jpg', '.jpeg']
    
    masks_folder = os.path.join(folder_path, 'masks')
    if not os.path.exists(masks_folder):
        raise FileNotFoundError(f"Masks folder not found: {masks_folder}")
    
    images = []
    for ext in file_extensions:
        pattern = os.path.join(masks_folder, f"*{ext}")
        files = glob.glob(pattern)
        for file_path in sorted(files):
            # Load mask as grayscale
            img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                images.append(img)
    
    return images

# Function to create camera intrinsics matrix
def create_intrinsics_matrix(fx: float, fy: float, cx: float, cy: float) -> np.ndarray:
    """
    Create a 3x3 camera intrinsics matrix.
    
    Args:
        fx: Focal length in x direction
        fy: Focal length in y direction
        cx: Principal point x coordinate
        cy: Principal point y coordinate
    
    Returns:
        3x3 intrinsics matrix
    """
    K = np.array([
        [fx, 0, cx],
        [0, fy, cy],
        [0, 0, 1]
    ], dtype=np.float64)
    return K


In [ ]:
# Load RGB, depth, and mask images
print(f"Loading data from: {data_path}")

try:
    rgb_images = load_rgb_images(data_path)
    print(f"Loaded {len(rgb_images)} RGB images")
except FileNotFoundError as e:
    print(f"Warning: {e}")
    rgb_images = []

try:
    depth_images = load_depth_images(data_path)
    print(f"Loaded {len(depth_images)} depth images")
except FileNotFoundError as e:
    print(f"Warning: {e}")
    depth_images = []

try:
    mask_images = load_mask_images(data_path)
    print(f"Loaded {len(mask_images)} mask images")
except FileNotFoundError as e:
    print(f"Warning: {e}")
    mask_images = []

if len(rgb_images) == 0 and len(depth_images) == 0:
    raise ValueError("No images loaded! Check data_path.")

# Determine image dimensions for intrinsics
if len(rgb_images) > 0:
    H, W = rgb_images[0].shape[:2]
    print(f"Image dimensions: {W}x{H}")
elif len(depth_images) > 0:
    H, W = depth_images[0].shape[:2]
    print(f"Image dimensions: {W}x{H}")

# Create camera intrinsics (using center of image as principal point)
cx, cy = W / 2.0, H / 2.0
cam_intrinsics = create_intrinsics_matrix(DEFAULT_FX, DEFAULT_FY, cx, cy)
print(f"\nCamera intrinsics:")
print(cam_intrinsics)
print(f"\nDepth factor: {DEFAULT_DEPTH_FACTOR} (assuming depth in mm)")


In [ ]:
# Select an image pair to process
image_idx = 0  # Change this to test different images

if len(depth_images) == 0:
    raise ValueError("No depth images available!")

if image_idx >= len(depth_images):
    image_idx = len(depth_images) - 1
    print(f"Warning: image_idx out of range, using last image (index {image_idx})")

# Get depth image
depth = depth_images[image_idx].copy()
print(f"\nProcessing image {image_idx}")
print(f"Depth image shape: {depth.shape}")
print(f"Depth image dtype: {depth.dtype}")
print(f"Depth range: [{depth.min()}, {depth.max()}]")
print(f"Valid depth pixels: {np.count_nonzero(depth > 0)} / {depth.size}")

# Get RGB image if available
rgb = None
if len(rgb_images) > image_idx:
    rgb = rgb_images[image_idx]
    print(f"RGB image shape: {rgb.shape}")
else:
    print("No RGB image available for this index")

# Get mask image if available
mask = None
if len(mask_images) > image_idx:
    mask = mask_images[image_idx]
    print(f"Mask image shape: {mask.shape}")
    print(f"Mask value range: [{mask.min()}, {mask.max()}]")
    # Convert mask to binary (threshold at 128 if grayscale)
    if mask.max() > 1:
        mask_binary = (mask > 128).astype(np.uint8)
    else:
        mask_binary = mask.astype(np.uint8)
    print(f"Mask pixels: {np.count_nonzero(mask_binary)} / {mask_binary.size}")
else:
    print("No mask image available for this index")

# Display depth image
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
if rgb is not None:
    axes[0].imshow(rgb)
    axes[0].set_title(f"RGB Image {image_idx}")
    axes[0].axis('off')
else:
    axes[0].text(0.5, 0.5, "No RGB image", ha='center', va='center')
    axes[0].axis('off')

# Normalize depth for visualization
depth_viz = depth.astype(np.float32)
if depth_viz.max() > 0:
    depth_viz = depth_viz / depth_viz.max()
axes[1].imshow(depth_viz, cmap='jet')
axes[1].set_title(f"Depth Image {image_idx}")
axes[1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Compute normals from depth
print("Computing normals from depth...")
print(f"Parameters:")
print(f"  depth_factor: {DEFAULT_DEPTH_FACTOR}")
print(f"  min_depth: {MIN_DEPTH} m")
print(f"  max_depth: {MAX_DEPTH} m")
print(f"  fill_missing_depth: {FILL_MISSING_DEPTH}")
if FILL_MISSING_DEPTH:
    print(f"  window_size: {WINDOW_SIZE}")
    print(f"  min_neighbors: {MIN_NEIGHBORS}")

normals, points_3d = compute_normals_from_depth(
    depth=depth,
    depth_factor=DEFAULT_DEPTH_FACTOR,
    cam_intrinsics=cam_intrinsics,
    min_depth=MIN_DEPTH,
    max_depth=MAX_DEPTH,
    fill_missing_depth=FILL_MISSING_DEPTH,
    window_size=WINDOW_SIZE,
    min_neighbors=MIN_NEIGHBORS,
)

print(f"\nComputed normals shape: {normals.shape}")
print(f"Computed points shape: {points_3d.shape}")

# Count valid normals
valid_normals = np.linalg.norm(normals, axis=-1) > 0.1  # Normals with non-zero magnitude
num_valid = np.count_nonzero(valid_normals)
print(f"Valid normals: {num_valid} / {normals.size // 3}")
if num_valid > 0:
    valid_normal_magnitudes = np.linalg.norm(normals[valid_normals], axis=-1)
    print(f"Normal magnitude range: [{valid_normal_magnitudes.min():.4f}, {valid_normal_magnitudes.max():.4f}]")
else:
    print("Warning: No valid normals found!")


In [ ]:
# Compare with Open3D normal computation
if HAS_OPEN3D:
    print("=" * 60)
    print("Comparing with Open3D normal computation")
    print("=" * 60)
    
    def compute_normals_open3d(depth, intrinsics, depth_scale):
        """
        Compute normals using Open3D.
        
        Args:
            depth: (H, W) depth image in raw units
            intrinsics: (3, 3) camera intrinsics matrix
            depth_scale: Scale factor to convert depth to meters
        
        Returns:
            normals: (H, W, 3) normal vectors
            points: (H, W, 3) 3D points
        """
        # Convert depth to meters
        depth_m = depth.astype(np.float32) * depth_scale
        
        # Create Open3D images
        depth_o3d = o3d.geometry.Image(depth_m)
        
        # Create camera intrinsic
        # Open3D expects: width, height, fx, fy, cx, cy
        H, W = depth.shape
        intrinsic = o3d.camera.PinholeCameraIntrinsic(
            W, H,
            intrinsics[0, 0], intrinsics[1, 1],
            intrinsics[0, 2], intrinsics[1, 2]
        )
        
        # Create point cloud from depth image
        pcd = o3d.geometry.PointCloud.create_from_depth_image(depth_o3d, intrinsic)
        
        # Estimate normals
        pcd.estimate_normals()
        pcd.orient_normals_consistent_tangent_plane(10)
        
        # Get normals and points
        normals_o3d = np.asarray(pcd.normals)
        points_o3d = np.asarray(pcd.points)
        
        # Reshape to (H, W, 3) format
        # Open3D's create_from_depth_image returns points in row-major order (y, x)
        # So we can reshape directly
        if len(points_o3d) == H * W:
            normals_reshaped = normals_o3d.reshape(H, W, 3)
            points_reshaped = points_o3d.reshape(H, W, 3)
        else:
            # If not all points are valid, we need to map them back
            # This happens when some depth values are invalid
            normals_reshaped = np.zeros((H, W, 3), dtype=np.float32)
            points_reshaped = np.zeros((H, W, 3), dtype=np.float32)
            
            # Create a mapping by projecting points back to image coordinates
            fx, fy = intrinsics[0, 0], intrinsics[1, 1]
            cx, cy = intrinsics[0, 2], intrinsics[1, 2]
            
            for i in range(len(points_o3d)):
                p = points_o3d[i]
                if p[2] > 0:  # Valid depth
                    # Project back to image coordinates
                    x = int((p[0] * fx / p[2]) + cx)
                    y = int((p[1] * fy / p[2]) + cy)
                    if 0 <= x < W and 0 <= y < H:
                        normals_reshaped[y, x] = normals_o3d[i]
                        points_reshaped[y, x] = p
        
        return normals_reshaped, points_reshaped
    
    # Compute normals using Open3D
    print("Computing normals with Open3D...")
    try:
        normals_o3d, points_3d_o3d = compute_normals_open3d(
            depth, cam_intrinsics, HO3D_DEPTH_SCALE
        )
        print(f"Open3D normals shape: {normals_o3d.shape}")
        print(f"Open3D points shape: {points_3d_o3d.shape}")
        
        # Count valid normals
        valid_normals_o3d = np.linalg.norm(normals_o3d, axis=-1) > 0.1
        num_valid_o3d = np.count_nonzero(valid_normals_o3d)
        print(f"Open3D valid normals: {num_valid_o3d} / {normals_o3d.size // 3}")
        
        if num_valid_o3d > 0:
            valid_normal_magnitudes_o3d = np.linalg.norm(normals_o3d[valid_normals_o3d], axis=-1)
            print(f"Open3D normal magnitude range: [{valid_normal_magnitudes_o3d.min():.4f}, {valid_normal_magnitudes_o3d.max():.4f}]")
        
        # Compare statistics
        print("\n" + "=" * 60)
        print("Comparison Statistics:")
        print("=" * 60)
        print(f"Custom method valid normals: {num_valid}")
        print(f"Open3D method valid normals: {num_valid_o3d}")
        
        # Compare where both are valid
        both_valid = valid_normals & valid_normals_o3d
        num_both_valid = np.count_nonzero(both_valid)
        print(f"Both methods valid: {num_both_valid}")
        
        if num_both_valid > 0:
            # Compute angular difference
            normals_custom_valid = normals[both_valid]
            normals_o3d_valid = normals_o3d[both_valid]
            
            # Normalize (should already be normalized, but ensure)
            normals_custom_norm = normals_custom_valid / (np.linalg.norm(normals_custom_valid, axis=1, keepdims=True) + 1e-8)
            normals_o3d_norm = normals_o3d_valid / (np.linalg.norm(normals_o3d_valid, axis=1, keepdims=True) + 1e-8)
            
            # Dot product (cosine of angle)
            dots = np.clip(np.sum(normals_custom_norm * normals_o3d_norm, axis=1), -1.0, 1.0)
            angles = np.arccos(dots) * 180.0 / np.pi  # Convert to degrees
            
            print(f"\nAngular difference statistics:")
            print(f"  Mean angle: {angles.mean():.2f} degrees")
            print(f"  Std angle:  {angles.std():.2f} degrees")
            print(f"  Min angle:  {angles.min():.2f} degrees")
            print(f"  Max angle:  {angles.max():.2f} degrees")
            print(f"  Median angle: {np.median(angles):.2f} degrees")
            
            # Histogram of angular differences
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            axes[0].hist(angles, bins=50, edgecolor='black', alpha=0.7)
            axes[0].set_xlabel('Angular Difference (degrees)')
            axes[0].set_ylabel('Frequency')
            axes[0].set_title('Distribution of Angular Differences')
            axes[0].grid(True, alpha=0.3)
            axes[0].axvline(angles.mean(), color='red', linestyle='--', label=f'Mean: {angles.mean():.2f}°')
            axes[0].legend()
            
            # Scatter plot: angle vs normal magnitude
            magnitudes_custom = np.linalg.norm(normals_custom_valid, axis=1)
            axes[1].scatter(magnitudes_custom, angles, s=1, alpha=0.3)
            axes[1].set_xlabel('Custom Normal Magnitude')
            axes[1].set_ylabel('Angular Difference (degrees)')
            axes[1].set_title('Angular Difference vs Normal Magnitude')
            axes[1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
        
        # Visual comparison
        print("\n" + "=" * 60)
        print("Visual Comparison:")
        print("=" * 60)
        
        # Side-by-side normal visualization
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # Row 1: Custom method
        normals_custom_rgb = (normals + 1.0) / 2.0
        normals_custom_rgb = np.clip(normals_custom_rgb, 0, 1)
        valid_mask_custom = np.linalg.norm(normals, axis=-1) > 0.1
        normals_custom_rgb[~valid_mask_custom] = 0.0
        
        axes[0, 0].imshow(normals_custom_rgb)
        axes[0, 0].set_title(f"Custom Method Normals ({num_valid} valid)")
        axes[0, 0].axis('off')
        
        # X component
        axes[0, 1].imshow(normals[:, :, 0], cmap='RdBu', vmin=-1, vmax=1)
        axes[0, 1].set_title("Custom: X Component")
        axes[0, 1].axis('off')
        plt.colorbar(axes[0, 1].images[0], ax=axes[0, 1], fraction=0.046)
        
        # Z component
        axes[0, 2].imshow(normals[:, :, 2], cmap='RdBu', vmin=-1, vmax=1)
        axes[0, 2].set_title("Custom: Z Component")
        axes[0, 2].axis('off')
        plt.colorbar(axes[0, 2].images[0], ax=axes[0, 2], fraction=0.046)
        
        # Row 2: Open3D method
        normals_o3d_rgb = (normals_o3d + 1.0) / 2.0
        normals_o3d_rgb = np.clip(normals_o3d_rgb, 0, 1)
        valid_mask_o3d = np.linalg.norm(normals_o3d, axis=-1) > 0.1
        normals_o3d_rgb[~valid_mask_o3d] = 0.0
        
        axes[1, 0].imshow(normals_o3d_rgb)
        axes[1, 0].set_title(f"Open3D Method Normals ({num_valid_o3d} valid)")
        axes[1, 0].axis('off')
        
        # X component
        axes[1, 1].imshow(normals_o3d[:, :, 0], cmap='RdBu', vmin=-1, vmax=1)
        axes[1, 1].set_title("Open3D: X Component")
        axes[1, 1].axis('off')
        plt.colorbar(axes[1, 1].images[0], ax=axes[1, 1], fraction=0.046)
        
        # Z component
        axes[1, 2].imshow(normals_o3d[:, :, 2], cmap='RdBu', vmin=-1, vmax=1)
        axes[1, 2].set_title("Open3D: Z Component")
        axes[1, 2].axis('off')
        plt.colorbar(axes[1, 2].images[0], ax=axes[1, 2], fraction=0.046)
        
        plt.suptitle("Normal Computation Comparison", fontsize=16)
        plt.tight_layout()
        plt.show()
        
        # Difference visualization
        if num_both_valid > 0:
            # Compute difference map
            diff_map = np.zeros((H, W), dtype=np.float32)
            diff_map[both_valid] = angles
            
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            
            im = axes[0].imshow(diff_map, cmap='hot', vmin=0, vmax=90)
            axes[0].set_title("Angular Difference Map (degrees)")
            axes[0].axis('off')
            plt.colorbar(im, ax=axes[0], fraction=0.046)
            
            # Overlay on RGB if available
            if rgb is not None:
                overlay_diff = diff_map.copy()
                overlay_diff[~both_valid] = np.nan
                axes[1].imshow(rgb)
                im2 = axes[1].imshow(overlay_diff, cmap='hot', alpha=0.6, vmin=0, vmax=90)
                axes[1].set_title("Angular Difference Overlay on RGB")
                axes[1].axis('off')
                plt.colorbar(im2, ax=axes[1], fraction=0.046)
            else:
                axes[1].axis('off')
            
            plt.tight_layout()
            plt.show()
        
    except Exception as e:
        print(f"Error computing normals with Open3D: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Open3D not available. Skipping comparison.")


In [ ]:
# Visualize normals as 2D images
def visualize_normals_2d(normals: np.ndarray, rgb: Optional[np.ndarray] = None, 
                        title: str = "Normal Visualization"):
    """
    Visualize normals as 2D RGB images where R=X, G=Y, B=Z.
    
    Args:
        normals: (H, W, 3) normal vectors
        rgb: Optional RGB image for overlay
        title: Plot title
    """
    H, W = normals.shape[:2]
    
    # Convert normals to RGB visualization
    # Normalize to [0, 1] range: (normal + 1) / 2
    normals_rgb = (normals + 1.0) / 2.0  # Maps [-1, 1] to [0, 1]
    normals_rgb = np.clip(normals_rgb, 0, 1)
    
    # Mask out invalid normals (where magnitude is near zero)
    valid_mask = np.linalg.norm(normals, axis=-1) > 0.1
    normals_rgb[~valid_mask] = 0.0  # Set invalid normals to black
    
    # Create visualization
    if rgb is not None:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    else:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Row 1 or only row: RGB image, normal RGB, and individual components
    row_idx = 0
    
    if rgb is not None:
        axes[row_idx, 0].imshow(rgb)
        axes[row_idx, 0].set_title("RGB Image")
        axes[row_idx, 0].axis('off')
        
        axes[row_idx, 1].imshow(normals_rgb)
        axes[row_idx, 1].set_title("Normals (RGB: X=Red, Y=Green, Z=Blue)")
        axes[row_idx, 1].axis('off')
        
        # Overlay normals on RGB
        overlay = 0.5 * rgb.astype(np.float32) / 255.0 + 0.5 * normals_rgb
        overlay = np.clip(overlay, 0, 1)
        axes[row_idx, 2].imshow(overlay)
        axes[row_idx, 2].set_title("Normals Overlay on RGB")
        axes[row_idx, 2].axis('off')
        
        row_idx = 1
    else:
        axes[0].imshow(normals_rgb)
        axes[0].set_title("Normals (RGB: X=Red, Y=Green, Z=Blue)")
        axes[0].axis('off')
    
    # Individual normal components
    if rgb is not None:
        # X component (Red channel)
        axes[row_idx, 0].imshow(normals[:, :, 0], cmap='RdBu', vmin=-1, vmax=1)
        axes[row_idx, 0].set_title("Normal X Component")
        axes[row_idx, 0].axis('off')
        plt.colorbar(axes[row_idx, 0].images[0], ax=axes[row_idx, 0])
        
        # Y component (Green channel)
        axes[row_idx, 1].imshow(normals[:, :, 1], cmap='RdBu', vmin=-1, vmax=1)
        axes[row_idx, 1].set_title("Normal Y Component")
        axes[row_idx, 1].axis('off')
        plt.colorbar(axes[row_idx, 1].images[0], ax=axes[row_idx, 1])
        
        # Z component (Blue channel)
        axes[row_idx, 2].imshow(normals[:, :, 2], cmap='RdBu', vmin=-1, vmax=1)
        axes[row_idx, 2].set_title("Normal Z Component")
        axes[row_idx, 2].axis('off')
        plt.colorbar(axes[row_idx, 2].images[0], ax=axes[row_idx, 2])
    else:
        # X component
        axes[1].imshow(normals[:, :, 0], cmap='RdBu', vmin=-1, vmax=1)
        axes[1].set_title("Normal X Component")
        axes[1].axis('off')
        plt.colorbar(axes[1].images[0], ax=axes[1])
        
        # Y component
        axes[2].imshow(normals[:, :, 1], cmap='RdBu', vmin=-1, vmax=1)
        axes[2].set_title("Normal Y Component")
        axes[2].axis('off')
        plt.colorbar(axes[2].images[0], ax=axes[2])
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()

# Visualize normals
visualize_normals_2d(normals, rgb, title=f"Normal Visualization - Image {image_idx}")


In [ ]:
# Interactive 3D point cloud visualization with normals
def visualize_pointcloud_3d_plotly(points: np.ndarray, normals: np.ndarray, 
                                  rgb: Optional[np.ndarray] = None,
                                  mask: Optional[np.ndarray] = None,
                                  subsample: int = 1,
                                  normal_scale: float = 0.01):
    """
    Create an interactive 3D point cloud visualization with normals using Plotly.
    
    Args:
        points: (H, W, 3) 3D points
        normals: (H, W, 3) normal vectors
        rgb: Optional (H, W, 3) RGB image for coloring points
        mask: Optional (H, W) binary mask to filter points (only show points within mask)
        subsample: Subsample factor (1 = all points, 2 = every 2nd point, etc.)
        normal_scale: Scale factor for normal vectors in visualization
    """
    H, W = points.shape[:2]
    
    # Subsample points and normals
    points_sub = points[::subsample, ::subsample]
    normals_sub = normals[::subsample, ::subsample]
    
    # Subsample mask if provided
    if mask is not None:
        mask_sub = mask[::subsample, ::subsample]
        # Convert mask to binary if needed
        if mask_sub.max() > 1:
            mask_binary = (mask_sub > 128).astype(bool)
        else:
            mask_binary = (mask_sub > 0).astype(bool)
    else:
        mask_binary = None
    
    # Reshape to (N, 3)
    points_sub = points_sub.reshape(-1, 3)
    normals_sub = normals_sub.reshape(-1, 3)
    
    # Filter valid points (finite and non-zero normals)
    valid_mask = (
        np.isfinite(points_sub).all(axis=1) & 
        (np.linalg.norm(normals_sub, axis=1) > 0.1)
    )
    
    # Apply mask filter if provided
    if mask_binary is not None:
        mask_flat = mask_binary.ravel()
        valid_mask = valid_mask & mask_flat
        print(f"Mask filtering: {np.count_nonzero(mask_flat)} points in mask, {np.count_nonzero(valid_mask)} valid after filtering")
    
    points_valid = points_sub[valid_mask]
    normals_valid = normals_sub[valid_mask]
    
    print(f"Visualizing {len(points_valid)} points (subsampled by {subsample})")
    
    # Create figure
    fig = go.Figure()
    
    # Color points
    if rgb is not None:
        rgb_sub = rgb[::subsample, ::subsample].reshape(-1, 3)[valid_mask]
        colors = rgb_sub / 255.0  # Normalize to [0, 1]
        colors_str = [f'rgb({int(c[0]*255)},{int(c[1]*255)},{int(c[2]*255)})' for c in colors]
    else:
        # Color by Z depth
        z_values = points_valid[:, 2]
        z_normalized = (z_values - z_values.min()) / (z_values.max() - z_values.min() + 1e-8)
        colors_str = [px.colors.sample_colorscale("Viridis", [z])[0] for z in z_normalized]
    
    # Add point cloud
    fig.add_trace(go.Scatter3d(
        x=points_valid[:, 0],
        y=points_valid[:, 1],
        z=points_valid[:, 2],
        mode='markers',
        marker=dict(
            size=2,
            color=colors_str,
            opacity=0.8
        ),
        name='Point Cloud'
    ))
    
    # Add normal vectors as arrows (subsample further for clarity)
    normal_subsample = max(1, len(points_valid) // 3000)  # Show max 3000 normals
    points_normals = points_valid[::normal_subsample]
    normals_normals = normals_valid[::normal_subsample]
    
    # Use shorter scale for better visibility
    arrow_scale = normal_scale * 0.5  # Make arrows shorter
    normals_scaled = normals_normals * arrow_scale
    
    # Use Plotly Cone for arrow visualization
    # Cones point from start to end, so we need to position them at the end of the arrow
    arrow_starts = points_normals
    arrow_ends = points_normals + normals_scaled
    
    # Create cones (arrows) pointing in the direction of normals
    fig.add_trace(go.Cone(
        x=arrow_ends[:, 0],
        y=arrow_ends[:, 1],
        z=arrow_ends[:, 2],
        u=normals_scaled[:, 0],
        v=normals_scaled[:, 1],
        w=normals_scaled[:, 2],
        sizemode='absolute',
        sizeref=arrow_scale * 0.3,  # Arrow head size
        anchor='tip',
        colorscale='Reds',
        showscale=False,
        name='Normals'
    ))
    
    # Also add lines from points to arrow tips for better visibility
    for i in range(len(points_normals)):
        p = points_normals[i]
        n = normals_scaled[i]
        fig.add_trace(go.Scatter3d(
            x=[p[0], p[0] + n[0]],
            y=[p[1], p[1] + n[1]],
            z=[p[2], p[2] + n[2]],
            mode='lines',
            line=dict(color='red', width=2),
            showlegend=(i == 0),
            name='Normal Lines' if i == 0 else None,
            hoverinfo='skip'
        ))
    
    fig.update_layout(
        title='3D Point Cloud with Normals',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        width=1000,
        height=800
    )
    
    return fig

def visualize_pointcloud_3d_matplotlib(points: np.ndarray, normals: np.ndarray,
                                      rgb: Optional[np.ndarray] = None,
                                      mask: Optional[np.ndarray] = None,
                                      subsample: int = 4,
                                      normal_scale: float = 0.01):
    """
    Create a 3D point cloud visualization with normals using Matplotlib.
    
    Args:
        points: (H, W, 3) 3D points
        normals: (H, W, 3) normal vectors
        rgb: Optional (H, W, 3) RGB image for coloring points
        mask: Optional (H, W) binary mask to filter points (only show points within mask)
        subsample: Subsample factor (1 = all points, 2 = every 2nd point, etc.)
        normal_scale: Scale factor for normal vectors in visualization
    """
    H, W = points.shape[:2]
    
    # Subsample points and normals
    points_sub = points[::subsample, ::subsample]
    normals_sub = normals[::subsample, ::subsample]
    
    # Subsample mask if provided
    if mask is not None:
        mask_sub = mask[::subsample, ::subsample]
        # Convert mask to binary if needed
        if mask_sub.max() > 1:
            mask_binary = (mask_sub > 128).astype(bool)
        else:
            mask_binary = (mask_sub > 0).astype(bool)
    else:
        mask_binary = None
    
    # Reshape to (N, 3)
    points_sub = points_sub.reshape(-1, 3)
    normals_sub = normals_sub.reshape(-1, 3)
    
    # Filter valid points
    valid_mask = (
        np.isfinite(points_sub).all(axis=1) & 
        (np.linalg.norm(normals_sub, axis=1) > 0.1)
    )
    
    # Apply mask filter if provided
    if mask_binary is not None:
        mask_flat = mask_binary.ravel()
        valid_mask = valid_mask & mask_flat
        print(f"Mask filtering: {np.count_nonzero(mask_flat)} points in mask, {np.count_nonzero(valid_mask)} valid after filtering")
    
    points_valid = points_sub[valid_mask]
    normals_valid = normals_sub[valid_mask]
    
    print(f"Visualizing {len(points_valid)} points (subsampled by {subsample})")
    
    # Create figure
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Color points
    if rgb is not None:
        rgb_sub = rgb[::subsample, ::subsample].reshape(-1, 3)[valid_mask]
        colors = rgb_sub / 255.0
    else:
        # Color by Z depth
        z_values = points_valid[:, 2]
        colors = plt.cm.viridis((z_values - z_values.min()) / (z_values.max() - z_values.min() + 1e-8))
    
    # Plot points
    ax.scatter(points_valid[:, 0], points_valid[:, 1], points_valid[:, 2], 
               c=colors, s=1, alpha=0.6)
    
    # Plot normals as arrows (subsample further)
    normal_subsample = max(1, len(points_valid) // 1500)  # Show max 1500 normals
    points_normals = points_valid[::normal_subsample]
    normals_normals = normals_valid[::normal_subsample]
    
    # Use shorter scale for better visibility
    arrow_scale = normal_scale * 0.5  # Make arrows shorter
    normals_scaled = normals_normals * arrow_scale
    
    # Use quiver3d to create arrows
    ax.quiver(
        points_normals[:, 0],
        points_normals[:, 1],
        points_normals[:, 2],
        normals_scaled[:, 0],
        normals_scaled[:, 1],
        normals_scaled[:, 2],
        color='red',
        arrow_length_ratio=0.3,  # Arrow head length as fraction of arrow
        length=1.0,  # Length is already in normals_scaled
        normalize=False,
        alpha=0.7,
        linewidth=1.5
    )
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title('3D Point Cloud with Normals')
    
    plt.tight_layout()
    plt.show()

# Visualize 3D point cloud (filtered by mask if available)
if mask is not None:
    print("Using mask to filter 3D visualization...")
    # Use mask_binary if it was created above, otherwise create it
    if 'mask_binary' not in locals():
        if mask.max() > 1:
            mask_binary = (mask > 128).astype(np.uint8)
        else:
            mask_binary = mask.astype(np.uint8)
else:
    mask_binary = None
    print("No mask available - showing all points")

if HAS_PLOTLY:
    print("Creating interactive 3D visualization with Plotly...")
    fig = visualize_pointcloud_3d_plotly(
        points_3d, normals, rgb, mask=mask_binary,
        subsample=1,  # Can use 1 since we're filtering by mask
        normal_scale=0.01  # Scale factor for normal vectors (will be halved for arrows)
    )
    fig.show()
else:
    print("Creating 3D visualization with Matplotlib...")
    visualize_pointcloud_3d_matplotlib(
        points_3d, normals, rgb, mask=mask_binary,
        subsample=2,  # Subsample more for matplotlib
        normal_scale=0.01  # Scale factor for normal vectors (will be halved for arrows)
    )


In [ ]:
# Test with different parameters
print("=" * 60)
print("Testing with depth filling enabled")
print("=" * 60)

# Compute normals with depth filling
normals_filled, points_3d_filled = compute_normals_from_depth(
    depth=depth,
    depth_factor=DEFAULT_DEPTH_FACTOR,
    cam_intrinsics=cam_intrinsics,
    min_depth=MIN_DEPTH,
    max_depth=MAX_DEPTH,
    fill_missing_depth=True,  # Enable depth filling
    window_size=5,  # Larger window
    min_neighbors=3,  # Require more neighbors
)

valid_normals_filled = np.linalg.norm(normals_filled, axis=-1) > 0.1
num_valid_filled = np.count_nonzero(valid_normals_filled)

print(f"\nComparison:")
print(f"  Without filling: {num_valid} valid normals")
print(f"  With filling:    {num_valid_filled} valid normals")
print(f"  Improvement:     {num_valid_filled - num_valid} additional normals")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Without filling
normals_rgb1 = (normals + 1.0) / 2.0
normals_rgb1 = np.clip(normals_rgb1, 0, 1)
valid_mask1 = np.linalg.norm(normals, axis=-1) > 0.1
normals_rgb1[~valid_mask1] = 0.0
axes[0].imshow(normals_rgb1)
axes[0].set_title(f"Normals (No Filling) - {num_valid} valid")
axes[0].axis('off')

# With filling
normals_rgb2 = (normals_filled + 1.0) / 2.0
normals_rgb2 = np.clip(normals_rgb2, 0, 1)
valid_mask2 = np.linalg.norm(normals_filled, axis=-1) > 0.1
normals_rgb2[~valid_mask2] = 0.0
axes[1].imshow(normals_rgb2)
axes[1].set_title(f"Normals (With Filling) - {num_valid_filled} valid")
axes[1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Additional analysis: Normal statistics and quality metrics
print("Normal Statistics:")
print("=" * 60)

valid_normals_mask = np.linalg.norm(normals, axis=-1) > 0.1
normals_valid = normals[valid_normals_mask]

if len(normals_valid) > 0:
    # Normal magnitudes (should be close to 1.0)
    magnitudes = np.linalg.norm(normals_valid, axis=1)
    print(f"Normal magnitudes:")
    print(f"  Mean: {magnitudes.mean():.6f}")
    print(f"  Std:  {magnitudes.std():.6f}")
    print(f"  Min:  {magnitudes.min():.6f}")
    print(f"  Max:  {magnitudes.max():.6f}")
    
    # Normal component distributions
    print(f"\nNormal component ranges:")
    print(f"  X: [{normals_valid[:, 0].min():.3f}, {normals_valid[:, 0].max():.3f}]")
    print(f"  Y: [{normals_valid[:, 1].min():.3f}, {normals_valid[:, 1].max():.3f}]")
    print(f"  Z: [{normals_valid[:, 2].min():.3f}, {normals_valid[:, 2].max():.3f}]")
    
    # Histogram of normal components
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].hist(normals_valid[:, 0], bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Normal X Component')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of X Components')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(normals_valid[:, 1], bins=50, edgecolor='black', alpha=0.7, color='green')
    axes[1].set_xlabel('Normal Y Component')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of Y Components')
    axes[1].grid(True, alpha=0.3)
    
    axes[2].hist(normals_valid[:, 2], bins=50, edgecolor='black', alpha=0.7, color='blue')
    axes[2].set_xlabel('Normal Z Component')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('Distribution of Z Components')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Visualize normal directions on unit sphere (2D projection)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # XY projection
    axes[0].scatter(normals_valid[:, 0], normals_valid[:, 1], s=1, alpha=0.3)
    circle = plt.Circle((0, 0), 1, fill=False, color='red', linestyle='--')
    axes[0].add_patch(circle)
    axes[0].set_xlabel('Normal X')
    axes[0].set_ylabel('Normal Y')
    axes[0].set_title('Normal Directions (XY Projection)')
    axes[0].set_aspect('equal')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(-1.2, 1.2)
    axes[0].set_ylim(-1.2, 1.2)
    
    # XZ projection
    axes[1].scatter(normals_valid[:, 0], normals_valid[:, 2], s=1, alpha=0.3)
    circle = plt.Circle((0, 0), 1, fill=False, color='red', linestyle='--')
    axes[1].add_patch(circle)
    axes[1].set_xlabel('Normal X')
    axes[1].set_ylabel('Normal Z')
    axes[1].set_title('Normal Directions (XZ Projection)')
    axes[1].set_aspect('equal')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(-1.2, 1.2)
    axes[1].set_ylim(-1.2, 1.2)
    
    plt.tight_layout()
    plt.show()
else:
    print("No valid normals to analyze!")
